# Dummy User Logs Generator (50k users)

- **입력(메타)**: `artwork_assigned_v2_post.json`, `artists_summary_v2.json`, `category_centroids_topk.json`
- **출력**: `outputs_json/dummy_user_logs.jsonl` (+ `user_profiles.json`, `log_stats.json`)

> ✅ 실제 서비스용 `artwork_vector_norm.json`(artwork_id, artist_id, artwork_vector) 이 있으면  
> 아래 로더 부분에서 `ARTWORK_PATH`만 바꿔서 그대로 사용하면 됩니다. (여기서는 제공된 파일로 artwork_id 리스트/카테고리 매핑을 구성)


In [1]:
import json
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
from datetime import datetime, timedelta, timezone

# -------------------
# Paths
# -------------------
DATA_DIR = Path.cwd()
ARTWORK_PATH = DATA_DIR / "outputs_json" / "artwork_assigned_v2_post.json"
ARTISTS_SUMMARY_PATH = DATA_DIR / "outputs_json" / "artists_summary_v2.json"
CATEGORY_CENTROIDS_PATH = DATA_DIR / "outputs_json" / "category_centroids_topk.json"

OUT_DIR = Path("outputs_json")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_LOGS_FLAT = OUT_DIR / "user_logs_flat.jsonl"
OUT_LOGS_GROUPED = OUT_DIR / "user_logs_grouped.jsonl"
OUT_PROFILES = OUT_DIR / "user_profiles.json"
OUT_STATS = OUT_DIR / "log_stats.json"

# -------------------
# Global config
# -------------------
SEED = 42
rng = np.random.default_rng(SEED)

NOW = datetime.now(timezone.utc)
LOOKBACK_DAYS = 180

N_USERS = 30000
USER_SEGMENT_RATIO = {"cold": 0.10, "normal": 0.50, "heavy": 0.40}

# 유저당 최대 로그 수(최근 200개까지만 사용)
MAX_LOGS_PER_USER = 200

# -------------------
# Action sparsity (BASE)  ✅ 먼저 정의해야 함
# -------------------
# 목표: VIEW(100) : LIKE(10) : REVIEW(1), COMMENT는 더 희소하게
BASE_PROB = {
    "stay": 0.35,
    "like": 0.10,
    "comment": 0.008,
    "review": 0.004,
}

# 헤비유저는 상호작용이 조금 더 많게
SEGMENT_MULT = {
    "cold": 0.75,
    "normal": 1.00,
    "heavy": 1.25,
}

# -------------------
# Derived caps ✅ BASE_PROB 이후 계산
# -------------------
EXPECTED_EVENTS_PER_VIEW = (
    1.0
    + BASE_PROB["stay"]
    + BASE_PROB["like"]
    + BASE_PROB["comment"]
    + BASE_PROB["review"]
)

MAX_VIEWS_PER_USER = int(MAX_LOGS_PER_USER / EXPECTED_EVENTS_PER_VIEW)  # 대략 136

# -------------------
# User view modes
# -------------------
VIEW_MODE_RATIO = {
    "cold":   {"artwork": 0.40, "artist": 0.20, "category": 0.40},
    "normal": {"artwork": 0.50, "artist": 0.30, "category": 0.20},
    "heavy":  {"artwork": 0.35, "artist": 0.45, "category": 0.20},
}

# -------------------
# VIEW count ranges ✅ MAX_VIEWS_PER_USER 계산 후 정의
# -------------------
VIEW_COUNT_RANGE = {
    "cold": (3, 12),
    "normal": (20, 90),
    "heavy": (120, min(350, MAX_VIEWS_PER_USER)),
}

print("EXPECTED_EVENTS_PER_VIEW:", EXPECTED_EVENTS_PER_VIEW)
print("MAX_VIEWS_PER_USER:", MAX_VIEWS_PER_USER)
print("VIEW_COUNT_RANGE:", VIEW_COUNT_RANGE)

EXPECTED_EVENTS_PER_VIEW: 1.4640000000000002
MAX_VIEWS_PER_USER: 136
VIEW_COUNT_RANGE: {'cold': (3, 12), 'normal': (20, 90), 'heavy': (120, 136)}


In [2]:
# -------------------
# Load metadata (JSON / JSONL safe)
# -------------------
import json
from pathlib import Path
from collections import defaultdict
import numpy as np

def load_json_or_jsonl(path: Path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as f:
        # first non-space char to detect JSON vs JSONL
        head = f.read(2048).lstrip()
        f.seek(0)

        # Try normal JSON first
        if head.startswith("{") or head.startswith("["):
            try:
                return json.load(f)
            except json.JSONDecodeError:
                pass

        # Fallback: JSONL
        rows = []
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
        return rows

artworks = load_json_or_jsonl(ARTWORK_PATH)
artists_summary = load_json_or_jsonl(ARTISTS_SUMMARY_PATH)
cat_centroids = load_json_or_jsonl(CATEGORY_CENTROIDS_PATH)

print("artworks:", len(artworks))
print("artists_summary:", len(artists_summary))
print("cat_centroids:", len(cat_centroids))

# -------------------
# Build mappings (robust field fallbacks)
# -------------------
cat_to_artworks = defaultdict(list)
artist_to_artworks = defaultdict(list)
artwork_to_cat = {}
artwork_to_artist = {}
artwork_to_topk_genres = {}

for a in artworks:
    aid = a.get("artwork_id") or a.get("id")
    artist = a.get("artist_id") or a.get("artist")
    if aid is None:
        continue

    # category 필드명 fallback
    cat = (
        a.get("primary_genre")
        or a.get("category_id")
        or a.get("primary_category")
        or a.get("category")
        or "unknown"
    )

    # topk genre 필드 fallback
    topk = (
        a.get("topk_genres")
        or a.get("genre")
        or a.get("genres")
        or [cat]
    )
    if isinstance(topk, str):
        topk = [topk]
    if not isinstance(topk, list):
        topk = [cat]
    if len(topk) == 0:
        topk = [cat]

    cat_to_artworks[str(cat)].append(aid)

    if artist is not None:
        artist_to_artworks[str(artist)].append(aid)
        artwork_to_artist[aid] = str(artist)
    else:
        artwork_to_artist[aid] = "unknown"

    artwork_to_cat[aid] = str(cat)
    artwork_to_topk_genres[aid] = [str(x) for x in topk]

categories = sorted(cat_to_artworks.keys())
artists = sorted(artist_to_artworks.keys())
all_artwork_ids = list(artwork_to_cat.keys())

print("num categories:", len(categories))
print("num artists:", len(artists))
print("num artworks:", len(all_artwork_ids))

# Category popularity baseline = number of artworks in category
cat_sizes = np.array([len(cat_to_artworks[c]) for c in categories], dtype=np.float32)
cat_pop = cat_sizes / max(cat_sizes.sum(), 1.0)

print("cat_pop sum:", float(cat_pop.sum()))
print("example category:", categories[0], "count:", len(cat_to_artworks[categories[0]]))

artworks: 27702
artists_summary: 4500
cat_centroids: 107
num categories: 107
num artists: 4500
num artworks: 27702
cat_pop sum: 1.0
example category: category001 count: 6


In [3]:
import json
from pathlib import Path
import numpy as np

ARTWORK_VEC_PATH = Path("/home/j-i14e107/Image_classification/outputs_json/artwork_vector.json")

# ✅ 수정 1: 파일 로드 방식 변경 (Line-by-line -> Full Load)
print(f"Loading {ARTWORK_VEC_PATH}...")
with ARTWORK_VEC_PATH.open("r", encoding="utf-8") as f:
    try:
        # 대부분의 경우 이 방식(JSON Array)입니다.
        vec_rows = json.load(f)
    except json.JSONDecodeError:
        # 만약 JSONL이라면 여기서 처리
        f.seek(0)
        vec_rows = []
        for line in f:
            line = line.strip()
            if line:
                vec_rows.append(json.loads(line))

print("vector rows:", len(vec_rows))
assert len(vec_rows) > 0

vec_ids = []
vec_mat_list = []

for r in vec_rows:
    # ID 추출 (키가 artwork_id, id, artworkId 중 하나)
    aid = r.get("artwork_id") or r.get("id") or r.get("artworkId")
    
    # ✅ 수정 2: 벡터 키 이름 안전하게 확인 (artwork_vector가 유력)
    v = r.get("artwork_vector") or r.get("vector") or r.get("embedding")
    
    if aid is None or v is None:
        continue
        
    # ID에서 확장자 제거 (안전을 위해)
    clean_id = Path(str(aid)).stem
    
    vec_ids.append(clean_id)
    vec_mat_list.append(v)

print("parsed vec_ids:", len(vec_ids))
assert len(vec_ids) > 0

vec_ids = np.array(vec_ids, dtype=object)
vec_mat = np.asarray(vec_mat_list, dtype=np.float32)   # (N, 512)

# normalize
vec_mat = vec_mat / (np.linalg.norm(vec_mat, axis=1, keepdims=True) + 1e-12)

id_to_vidx = {aid: i for i, aid in enumerate(vec_ids)}

print("vec_mat:", vec_mat.shape, "| example id:", vec_ids[0])

Loading /home/j-i14e107/Image_classification/outputs_json/artwork_vector.json...
vector rows: 27702
parsed vec_ids: 27702
vec_mat: (27702, 512) | example id: category030_1137


In [4]:
print("len(vec_rows):", len(vec_rows))
print("len(vec_ids):", len(vec_ids))
print("vec_mat shape:", vec_mat.shape)
print("id_to_vidx size:", len(id_to_vidx))

len(vec_rows): 27702
len(vec_ids): 27702
vec_mat shape: (27702, 512)
id_to_vidx size: 27702


## 센트로이드 벡터를 여기서 '왜' 쓰나?

- **카테고리 고정 ID만으로도** (category → artworks 매핑이 있으면) *카테고리 기반 추천/로그 생성은 가능*합니다.  
- **센트로이드 벡터는** “그 카테고리 대표 분위기/스타일”을 수치화한 것이고, 아래에 특히 유용합니다:
  1) **카테고리 텍스트가 새로 들어올 때**(미등록/변형 단어) 임베딩으로 가장 비슷한 기존 카테고리로 매핑  
  2) **카테고리 간 유사도**(추천 확장, 연관 카테고리 탐색)  
  3) **콜드스타트**에서 “카테고리 벡터 ↔ 작품 벡터”로 바로 검색 (둘이 **같은 임베딩 공간**일 때)

> 이 더미 로그 생성에서는 “카테고리 중심 유저”가 선택한 카테고리에서 작품을 뽑기만 하면 되므로  
> **센트로이드는 필수는 아니고** “연관 카테고리 섞기” 같은 고급 샘플링 옵션에 쓰면 좋습니다.


In [5]:
# -------------------
# (Optional) category similarity from centroids
# - 같은 임베딩 공간이면 cosine 유사도로 연관 카테고리를 섞을 수 있음
# -------------------
centroid_by_cat = {d["category"]: np.asarray(d["centroid"], dtype=np.float32) for d in cat_centroids}
centroid_cats = sorted(centroid_by_cat.keys())
C = np.stack([centroid_by_cat[c] for c in centroid_cats], axis=0)  # (Nc, D)

# cosine sim
C_norm = C / (np.linalg.norm(C, axis=1, keepdims=True) + 1e-12)
cat_sim = C_norm @ C_norm.T  # (Nc, Nc)
np.fill_diagonal(cat_sim, -np.inf)

topk_sim_idx = np.argsort(-cat_sim, axis=1)[:, :5]
cat_sim_neighbors = {
    centroid_cats[i]: [centroid_cats[j] for j in topk_sim_idx[i]]
    for i in range(len(centroid_cats))
}

print("Example neighbors:", centroid_cats[0], "->", cat_sim_neighbors[centroid_cats[0]])


Example neighbors: category001 -> ['category106', 'category087', 'category010', 'category040', 'category049']


In [6]:
# -------------------
# Popularity weights (80/20 head-tail) with Zipf-like within head & tail
# -------------------
N = len(all_artwork_ids)
head_n = max(1, int(0.20 * N))
tail_n = N - head_n

# ranking: just fixed order by artwork_id for determinism; could also shuffle
ranked = list(all_artwork_ids)
ranked.sort()

head_items = ranked[:head_n]
tail_items = ranked[head_n:]

def zipf_weights(m, s=1.2):
    r = np.arange(1, m+1, dtype=np.float32)
    w = 1.0 / (r ** s)
    return w / w.sum()

head_w = zipf_weights(head_n, s=1.15)  # head는 더 완만하게
tail_w = zipf_weights(tail_n, s=1.35)  # tail은 더 가파르게

# mix: 80% mass to head, 20% to tail
global_items = np.array(head_items + tail_items, dtype=object)
global_w = np.concatenate([0.80 * head_w, 0.20 * tail_w]).astype(np.float64)
global_w = global_w / global_w.sum()

# quick sanity: expected head mass
print("Head mass:", float(global_w[:head_n].sum()))


Head mass: 0.7999999896401099


In [7]:
# -------------------
# User profile sampling
# -------------------
def sample_choice(prob_dict):
    keys = list(prob_dict.keys())
    probs = np.array([prob_dict[k] for k in keys], dtype=np.float64)
    probs = probs / probs.sum()
    return rng.choice(keys, p=probs)

def sample_k_from_categories(k, bias=cat_pop):
    # sample without replacement weighted by cat_pop
    k = min(k, len(categories))
    return list(rng.choice(categories, size=k, replace=False, p=bias))

# build artist->primary_categories from artists_summary (for preference coherence)
artist_primary_cats = {}
for a in artists_summary:
    artist_primary_cats[a["artist_id"]] = a.get("primary_categories", [])

def sample_preferred_artists(pref_cats, k):
    # 후보: pref_cats를 하나라도 포함하는 artist
    candidates = []
    for artist, cats in artist_primary_cats.items():
        if any(c in cats for c in pref_cats):
            candidates.append(artist)
    if not candidates:
        candidates = artists
    k = min(k, len(candidates))
    return list(rng.choice(candidates, size=k, replace=False))

def make_user_profile(uid, segment):
    # categories
    if segment == "cold":
        n_cat = int(rng.integers(1, 4))
        n_artist = int(rng.integers(0, 3))
    elif segment == "normal":
        n_cat = int(rng.integers(2, 6))
        n_artist = int(rng.integers(1, 5))
    else:
        n_cat = int(rng.integers(3, 9))
        n_artist = int(rng.integers(2, 8))
    pref_cats = sample_k_from_categories(n_cat)
    pref_artists = sample_preferred_artists(pref_cats, max(1, n_artist)) if n_artist > 0 else []

    view_mode = sample_choice(VIEW_MODE_RATIO[segment])
    return {
        "member_id": f"u_{uid:05d}",
        "segment": segment,
        "view_mode": view_mode,
        "preferred_categories": pref_cats,
        "preferred_artists": pref_artists,
    }

# Create all profiles
n_cold = int(N_USERS * USER_SEGMENT_RATIO["cold"])
n_normal = int(N_USERS * USER_SEGMENT_RATIO["normal"])
n_heavy = N_USERS - n_cold - n_normal

segments = (["cold"] * n_cold) + (["normal"] * n_normal) + (["heavy"] * n_heavy)
rng.shuffle(segments)

profiles = [make_user_profile(i, segments[i]) for i in range(N_USERS)]

print("Profiles made:", len(profiles))
print(Counter([p["segment"] for p in profiles]))
print(Counter([p["view_mode"] for p in profiles]))


Profiles made: 30000
Counter({'normal': 15000, 'heavy': 12000, 'cold': 3000})
Counter({np.str_('artwork'): 12915, np.str_('artist'): 10463, np.str_('category'): 6622})


In [8]:
import numpy as np
import torch
import math

# -------------------
# GPU setup (once)
# -------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# vec_mat: (N, D) numpy float32 (already normalized)
vec_mat_t = torch.from_numpy(vec_mat).to(device)  # (N, 512), float32 on GPU

# -------------------
# Event generation helpers
# -------------------
def recency_day_offset(scale=45.0, max_days=LOOKBACK_DAYS):
    x = rng.exponential(scale)
    return min(float(x), float(max_days))

def session_length_views(segment):
    if segment == "cold":
        return int(rng.integers(1, 4))
    if segment == "normal":
        return int(rng.integers(2, 8))
    return int(rng.integers(3, 12))

def dwell_seconds(segment):
    base = {"cold": 15, "normal": 25, "heavy": 35}[segment]
    x = rng.lognormal(mean=math.log(base), sigma=0.6)
    return int(max(10, min(240, x)))

def p_adjust(base_p, segment, matched_pref=False, long_dwell=False):
    p = base_p * SEGMENT_MULT[segment]
    if matched_pref: p *= 1.25
    if long_dwell: p *= 1.35
    return min(0.95, float(p))

# -------------------
# Global sampling (CPU rng.choice is fine)
# -------------------
global_items_arr = np.array(global_items, dtype=object)
global_w_arr = np.asarray(global_w, dtype=np.float64)
global_w_arr = global_w_arr / global_w_arr.sum()

def sample_artwork_global():
    return str(rng.choice(global_items_arr, p=global_w_arr))

# -------------------
# ✅ Category sampling cache (VERY IMPORTANT speed-up)
# -------------------
global_w_by_id = {str(a): float(global_w_arr[i]) for i, a in enumerate(global_items_arr)}

cat_items_np = {}
cat_probs_np = {}
for cat, items in cat_to_artworks.items():
    items2 = [str(it) for it in items if str(it) in global_w_by_id]
    if not items2:
        continue
    w = np.array([global_w_by_id[it] for it in items2], dtype=np.float64)
    w = w / w.sum()
    cat_items_np[cat] = np.array(items2, dtype=object)
    cat_probs_np[cat] = w

def sample_artwork_from_category(cat):
    arr = cat_items_np.get(cat)
    if arr is None or len(arr) == 0:
        return sample_artwork_global()
    w = cat_probs_np[cat]
    return str(rng.choice(arr, p=w))

def sample_artwork_from_artist(artist):
    items = artist_to_artworks.get(artist, [])
    if not items:
        return sample_artwork_global()
    return str(rng.choice(np.array(items, dtype=object)))

# -------------------
# ✅ GPU Vector similarity samplers
# -------------------
@torch.no_grad()
def _gpu_softmax_probs(sims: torch.Tensor, temp: float):
    # sims: (M,) on GPU
    logits = sims / max(temp, 1e-8)
    logits = logits - logits.max()
    return torch.softmax(logits, dim=0)

def sample_artwork_similar_to_vector(last_aid, candidate_pool=120, temp=0.07):
    """
    글로벌 후보(candidate_pool)에서 last_aid와 cosine 유사도가 높은 작품을 샘플링.
    유사도 계산은 GPU(torch)로 수행.
    """
    if last_aid not in id_to_vidx:
        return sample_artwork_global()

    # 후보는 CPU에서 뽑고(롱테일 유지), 인덱스만 GPU로 보냄
    cand = rng.choice(global_items_arr, size=min(candidate_pool, len(global_items_arr)), replace=False, p=global_w_arr)
    cand = np.array([c for c in cand if c in id_to_vidx], dtype=object)
    if len(cand) == 0:
        return sample_artwork_global()

    last_idx = int(id_to_vidx[last_aid])
    cand_idx = torch.tensor([int(id_to_vidx[c]) for c in cand], device=device, dtype=torch.long)

    v_last = vec_mat_t[last_idx]          # (D,)
    C = vec_mat_t.index_select(0, cand_idx)  # (M, D)
    sims = (C @ v_last).float()           # (M,)

    p_gpu = _gpu_softmax_probs(sims, temp)    # (M,) GPU
    p = p_gpu.cpu().numpy()                   # -> CPU for rng.choice

    return str(rng.choice(cand, p=p))

def sample_similar_within_category(last_aid, cat, candidate_pool=100, temp=0.08):
    """
    카테고리 내 후보에서 last_aid와 cosine 유사도가 높은 작품 샘플링.
    유사도 계산은 GPU(torch)로 수행.
    """
    if last_aid not in id_to_vidx:
        return sample_artwork_from_category(cat)

    items = [x for x in cat_to_artworks.get(cat, []) if x in id_to_vidx]
    if not items:
        return sample_artwork_from_category(cat)

    m = min(candidate_pool, len(items))
    cand = rng.choice(np.array(items, dtype=object), size=m, replace=False)

    last_idx = int(id_to_vidx[last_aid])
    cand_idx = torch.tensor([int(id_to_vidx[c]) for c in cand], device=device, dtype=torch.long)

    v_last = vec_mat_t[last_idx]
    C = vec_mat_t.index_select(0, cand_idx)
    sims = (C @ v_last).float()

    p_gpu = _gpu_softmax_probs(sims, temp)
    p = p_gpu.cpu().numpy()

    return str(rng.choice(cand, p=p))

# -------------------
# choose_next_artwork
# -------------------
def choose_next_artwork(profile, last_artwork=None):
    seg = profile["segment"]
    mode = profile["view_mode"]
    pref_cats = profile["preferred_categories"]
    pref_artists = profile["preferred_artists"]

    if mode == "category":
        cat = str(rng.choice(np.array(pref_cats, dtype=object))) if pref_cats else str(rng.choice(np.array(categories, dtype=object), p=cat_pop))

        if cat in cat_sim_neighbors and rng.random() < 0.15:
            nbrs = [c for c in cat_sim_neighbors[cat] if c in cat_to_artworks]
            if nbrs:
                cat = str(rng.choice(np.array([cat] + nbrs, dtype=object)))

        # ✅ 카테고리 유지 + 작화 유사도
        if last_artwork is not None and rng.random() < 0.35:
            return sample_similar_within_category(last_artwork, cat, candidate_pool=100, temp=0.08)

        return sample_artwork_from_category(cat)

    if mode == "artist" and pref_artists:
        artist = str(rng.choice(np.array(pref_artists, dtype=object)))
        return sample_artwork_from_artist(artist)

    # artwork 중심
    if last_artwork is None or rng.random() < 0.55:
        return sample_artwork_global()

    return sample_artwork_similar_to_vector(last_artwork, candidate_pool=120, temp=0.07)

In [9]:
a0 = str(vec_ids[0])
a1 = sample_artwork_similar_to_vector(a0, candidate_pool=500, temp=0.05)
sim = float(vec_mat_t[id_to_vidx[a0]] @ vec_mat_t[id_to_vidx[a1]])
print("a0:", a0, "a1:", a1, "cosine:", sim)
print("device:", vec_mat_t.device)

a0: category030_1137 a1: category010_0316 cosine: 0.735488772392273
device: cuda:0


In [10]:
import math
import time
import json
from collections import Counter
from datetime import timedelta

# ================================
# Outputs (flat events + grouped per-user)
# ================================
# - OUT_LOGS_FLAT: JSONL (1 line = 1 event)  -> 학습/디버깅용
# - OUT_LOGS_GROUPED: JSONL (1 line = 1 user) -> 서비스 입력 포맷(요구사항 반영)
#
# 서비스 입력(변경 후):
# {"member_id":"u_00000","timestamp":[{"artwork_id":"...","timestamp":"VIEW"}, ...]}
#
# ✅ NOTE: 위 포맷에서 inner "timestamp"는 '액션 타입' 의미로 사용합니다.
#         (실제 시각은 grouped 파일에 포함하지 않습니다.)
# ================================

for p in (OUT_LOGS_FLAT, OUT_LOGS_GROUPED):
    if p.exists():
        p.unlink()

total_events = 0
action_counter = Counter()

BATCH = []
BATCH_SIZE = 20_000
REPORT_EVERY_USERS = 500

t0 = time.time()

def dump_events_batch(f, batch):
    f.write("".join(json.dumps(r, ensure_ascii=False) + "\n" for r in batch))
    batch.clear()

with open(OUT_LOGS_FLAT, "a", encoding="utf-8") as f_flat, open(OUT_LOGS_GROUPED, "a", encoding="utf-8") as f_grouped:
    for i, profile in enumerate(profiles):
        seg = profile["segment"]

        # ---- how many VIEWS for this user (cap by MAX_VIEWS_PER_USER) ----
        n_views = int(rng.integers(VIEW_COUNT_RANGE[seg][0], VIEW_COUNT_RANGE[seg][1] + 1))
        n_views = min(n_views, MAX_VIEWS_PER_USER)

        # ---- build chronological sessions (older -> newer) ----
        # session count is derived from view count and random session lengths
        views_left = n_views
        sessions = []
        while views_left > 0:
            v_in_session = min(views_left, session_length_views(seg))
            views_left -= v_in_session
            day_off = recency_day_offset(scale=45.0)
            start = NOW - timedelta(days=day_off) - timedelta(seconds=float(rng.integers(0, 86400)))
            sessions.append((start, v_in_session))

        # sort sessions by start time (chronological)
        sessions.sort(key=lambda x: x[0])

        user_events = []  # accumulate per-user (<= MAX_LOGS_PER_USER)
        last_art = None

        for start, v_in_session in sessions:
            t = start
            for _ in range(v_in_session):
                if len(user_events) >= MAX_LOGS_PER_USER:
                    break

                aid = choose_next_artwork(profile, last_artwork=last_art)
                last_art = aid

                # VIEW
                ev_view = {
                    "member_id": profile["member_id"],
                    "artwork_id": aid,
                    "action_type": "VIEW",
                    "timestamp": t.isoformat(),
                }
                user_events.append(ev_view)

                matched_pref = (artwork_to_cat.get(aid) in set(profile["preferred_categories"])) or \
                               (artwork_to_artist.get(aid) in set(profile["preferred_artists"]))

                # STAY (optional)
                dw = dwell_seconds(seg)
                long_dw = dw >= 45

                t_end = t  # track last time within this view-block
                t_stay = t
                if len(user_events) < MAX_LOGS_PER_USER and rng.random() < p_adjust(BASE_PROB["stay"], seg, matched_pref=matched_pref, long_dwell=long_dw):
                    t_stay = t + timedelta(seconds=dw)
                    user_events.append({
                        "member_id": profile["member_id"],
                        "artwork_id": aid,
                        "action_type": "STAY",
                        "timestamp": t_stay.isoformat(),
                        "stay_seconds": int(dw),
                    })
                    t_end = max(t_end, t_stay)

                # LIKE (optional)
                liked = False
                t_like = None
                if len(user_events) < MAX_LOGS_PER_USER and rng.random() < p_adjust(BASE_PROB["like"], seg, matched_pref=matched_pref, long_dwell=long_dw):
                    t_like = t_end + timedelta(seconds=int(rng.integers(3, 60)))
                    user_events.append({
                        "member_id": profile["member_id"],
                        "artwork_id": aid,
                        "action_type": "LIKE",
                        "timestamp": t_like.isoformat(),
                    })
                    liked = True
                    t_end = max(t_end, t_like)

                # COMMENT (optional)
                if len(user_events) < MAX_LOGS_PER_USER and rng.random() < p_adjust(BASE_PROB["comment"], seg, matched_pref=matched_pref, long_dwell=long_dw) * (1.4 if liked else 1.0):
                    t_c = t_end + timedelta(seconds=int(rng.integers(10, 120)))
                    user_events.append({
                        "member_id": profile["member_id"],
                        "artwork_id": aid,
                        "action_type": "COMMENT",
                        "timestamp": t_c.isoformat(),
                    })
                    t_end = max(t_end, t_c)

                # REVIEW (optional)
                if len(user_events) < MAX_LOGS_PER_USER and rng.random() < p_adjust(BASE_PROB["review"], seg, matched_pref=matched_pref, long_dwell=long_dw) * (1.2 if liked else 1.0):
                    t_r = t_end + timedelta(seconds=int(rng.integers(30, 600)))
                    user_events.append({
                        "member_id": profile["member_id"],
                        "artwork_id": aid,
                        "action_type": "REVIEW",
                        "timestamp": t_r.isoformat(),
                    })
                    t_end = max(t_end, t_r)

                # next view time MUST be after the last event time of this view-block
                gap = timedelta(seconds=int(rng.integers(20, 360)))
                t = t_end + gap

            if len(user_events) >= MAX_LOGS_PER_USER:
                break

        # ---- update global counters + write flat JSONL (batched) ----
        for e in user_events:
            action_counter[e["action_type"]] += 1
        total_events += len(user_events)

        BATCH.extend(user_events)
        if len(BATCH) >= BATCH_SIZE:
            dump_events_batch(f_flat, BATCH)

        # ---- write grouped JSONL (latest-first) ----
        # sort by real timestamp descending, then drop real time in grouped format
        user_events_sorted = sorted(user_events, key=lambda x: x["timestamp"], reverse=True)

        grouped = {
            "member_id": profile["member_id"],
            "timestamp": [
                {"artwork_id": e["artwork_id"], "timestamp": e["action_type"]}
                for e in user_events_sorted
            ]
        }
        f_grouped.write(json.dumps(grouped, ensure_ascii=False) + "\n")

        # ---- progress ----
        if (i + 1) % REPORT_EVERY_USERS == 0:
            f_flat.flush(); f_grouped.flush()
            dt = time.time() - t0
            print(f"[{i+1}/{len(profiles)} users] events={total_events:,} elapsed={dt:.1f}s dist={dict(action_counter)}")

    # final flush
    if BATCH:
        dump_events_batch(f_flat, BATCH)
        f_flat.flush()
    f_grouped.flush()

print("Saved flat logs :", OUT_LOGS_FLAT)
print("Saved grouped   :", OUT_LOGS_GROUPED)
print("Total events    :", total_events)
print("Action dist     :", action_counter)

[500/30000 users] events=61,795 elapsed=8.9s dist={'VIEW': 36923, 'STAY': 18692, 'LIKE': 5408, 'REVIEW': 533, 'COMMENT': 239}
[1000/30000 users] events=121,816 elapsed=18.0s dist={'VIEW': 72787, 'STAY': 36903, 'LIKE': 10590, 'REVIEW': 1080, 'COMMENT': 456}
[1500/30000 users] events=184,581 elapsed=27.6s dist={'VIEW': 110435, 'STAY': 55752, 'LIKE': 16016, 'REVIEW': 1675, 'COMMENT': 703}
[2000/30000 users] events=247,161 elapsed=37.0s dist={'VIEW': 147729, 'STAY': 74836, 'LIKE': 21422, 'REVIEW': 2257, 'COMMENT': 917}
[2500/30000 users] events=308,682 elapsed=46.3s dist={'VIEW': 184520, 'STAY': 93469, 'LIKE': 26712, 'REVIEW': 2828, 'COMMENT': 1153}
[3000/30000 users] events=370,316 elapsed=55.7s dist={'VIEW': 221347, 'STAY': 112132, 'LIKE': 32063, 'REVIEW': 3379, 'COMMENT': 1395}
[3500/30000 users] events=431,800 elapsed=64.5s dist={'VIEW': 258100, 'STAY': 130799, 'LIKE': 37369, 'REVIEW': 3930, 'COMMENT': 1602}
[4000/30000 users] events=495,583 elapsed=74.6s dist={'VIEW': 296157, 'STAY': 

In [11]:
# -------------------
# Save profiles + stats
# -------------------
with open(OUT_PROFILES, "w", encoding="utf-8") as f:
    json.dump(profiles, f, ensure_ascii=False, indent=2)

# 80/20 sanity check on VIEW only
view_counts = Counter()
with open(OUT_LOGS_FLAT, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        if e["action_type"] == "VIEW":
            view_counts[e["artwork_id"]] += 1

sorted_views = sorted(view_counts.items(), key=lambda x: -x[1])
topk = max(1, int(0.20 * len(sorted_views)))
top_views = sum(v for _, v in sorted_views[:topk])
all_views = sum(v for _, v in sorted_views)

stats = {
    "created_at": NOW.isoformat(),
    "n_users": N_USERS,
    "n_artworks": len(all_artwork_ids),
    "n_artists": len(artists),
    "n_categories": len(categories),
    "total_events": int(total_events),
    "action_distribution": {k: int(v) for k, v in action_counter.items()},
    "view_80_20_ratio": float(top_views / max(1, all_views)),
    "top20pct_items_count": int(topk),
    "unique_viewed_items": int(len(sorted_views)),
}

with open(OUT_STATS, "w", encoding="utf-8") as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print("Saved profiles:", OUT_PROFILES)
print("Saved stats:", OUT_STATS)
print("VIEW 80/20 (top20% share):", stats["view_80_20_ratio"])


Saved profiles: outputs_json/user_profiles.json
Saved stats: outputs_json/log_stats.json
VIEW 80/20 (top20% share): 0.572377123297247


In [12]:
# -------------------
# Quick check snippet (fixing your Counter code)
# -------------------
import json
from collections import Counter

path = str(OUT_LOGS_FLAT)

cnt_action = Counter()
cnt_user = Counter()
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        e = json.loads(line)
        cnt_action[e["action_type"]] += 1
        cnt_user[e["member_id"]] += 1

avg = sum(cnt_user.values()) / max(1, len(cnt_user))
print("Action distribution:", cnt_action)
print("Users:", len(cnt_user), "Avg events/user:", avg)


Action distribution: Counter({'VIEW': 2217365, 'STAY': 1123515, 'LIKE': 321238, 'REVIEW': 33001, 'COMMENT': 13576})
Users: 30000 Avg events/user: 123.62316666666666


## (Deprecated)
이 셀은 과거에 `action_type`을 `timestamp` 필드에 덮어써서 저장하는 변환을 수행했었습니다.

**현재 서비스 입력 스키마 변경(유저별 그룹 + 최신순 + inner `timestamp`=action_type)에 맞지 않고,**
학습에서도 액션/시간 정보가 붕괴되므로 사용하지 않습니다.

대신 위의 생성 셀에서 `user_logs_flat.jsonl`(학습/디버깅용)과 `user_logs_grouped.jsonl`(서비스 입력용)을
동시에 생성합니다.
